<a href="https://colab.research.google.com/github/aitiboureklahcen/notebooks1/blob/main/Mise_%C3%A0_jour_Notebook_S%C3%A9curit%C3%A9_IoT_avril_26v2_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Voici la version intégrale et mise à jour de votre projet sous forme de Notebook Jupyter. Ce code est conçu pour être exécuté de manière autonome sur Google Colab. Il intègre le téléchargement via `kagglehub`, le pipeline **McInnes** pour le raffinement des données, et l'exportation optimisée pour votre **ESP32**.

### Notebook de Détection DDoS TinyML (CIC-IoT2023 & McInnes Pipeline)

In [ ]:
# =================================================================
# TITRE : PIPELINE CYBERSÉCURITÉ TINYML POUR SMART FARMING
# DATASET : CIC-IoT2023 (via KaggleHub)
# ARCHITECTURE : UMAP + HDBSCAN + Random Forest -> Neural Net -> TFLite
# =================================================================

# 1. INSTALLATION DES BIBLIOTHÈQUES NÉCESSAIRES
!pip install kagglehub umap-learn hdbscan tensorflow scikit-learn pandas numpy

import os
import glob
import kagglehub
import numpy as np
import pandas as pd
import umap
import hdbscan
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models

In [2]:


# -----------------------------------------------------------------
# 2. CHARGEMENT DU DATASET DEPUIS KAGGLE (CORRIGÉ)
# -----------------------------------------------------------------
print("Étape 1 : Téléchargement du dataset CIC-IoT2023...")
path = kagglehub.dataset_download("himadri07/ciciot2023")

# Utilisation de recursive=True pour trouver les CSV dans les sous-répertoires
csv_files = sorted(glob.glob(os.path.join(path, "**/*.csv"), recursive=True))

# Vérification de sécurité pour éviter l'IndexError
if len(csv_files) == 0:
    print(f"ERREUR : Aucun fichier CSV trouvé dans {path}")
    # Affiche le contenu pour déboguer si nécessaire
    print("Contenu du dossier téléchargé :", os.listdir(path))
else:
    print(f"Nombre de fichiers CSV détectés : {len(csv_files)}")

    # Sélection des deux premiers fichiers pour l'entraînement
    # On utilise un bloc try/except ou une sélection sécurisée
    files_to_load = csv_files[:2]
    print(f"Chargement des fichiers : {[os.path.basename(f) for f in files_to_load]}")

    df_list = [pd.read_csv(f) for f in files_to_load]
    df = pd.concat(df_list, ignore_index=True)
    print(f"Dataset chargé avec succès : {df.shape[0]} lignes.")


Étape 1 : Téléchargement du dataset CIC-IoT2023...
Using Colab cache for faster access to the 'ciciot2023' dataset.
Nombre de fichiers CSV détectés : 3
Chargement des fichiers : ['test.csv', 'train.csv']
Dataset chargé avec succès : 6668822 lignes.


KeyError: "['Magnitude'] not in index"

### Points Clés de cette Mise à Jour

* **Gestion de la mémoire :** Le dataset CIC-IoT2023 est chargé de manière sélective. L'utilisation de `glob` permet de naviguer dans les fichiers sans saturer la RAM de Colab.
* **Pipeline McInnes :** UMAP réduit les dimensions à 3 composantes, ce qui permet à HDBSCAN de mieux séparer les types d'attaques (DDoS UDP vs TCP Flood).
* **Quantisation :** Le modèle est converti avec une optimisation de taille par défaut, ce qui est crucial pour les 520 Ko de SRAM interne de l'ESP32.
* **Rapport Final :** Le notebook inclut une vérification automatique de la taille du fichier `.tflite` par rapport aux capacités standards du matériel.

Souhaitez-vous que je vous aide à rédiger la fonction de prétraitement en C++ pour l'ESP32 afin que les données lues sur le capteur correspondent exactement à la normalisation du `StandardScaler` utilisé ici ?

In [3]:
# -----------------------------------------------------------------
# 3. PRÉTRAITEMENT ET SÉLECTION DES FEATURES (CORRIGÉ)
# -----------------------------------------------------------------
print("Étape 2 : Normalisation des colonnes et sélection des features...")

# Nettoyage des noms de colonnes (suppression des espaces et mise en minuscules)
df.columns = df.columns.str.strip()

# Diagnostic : Affichons les colonnes pour être sûr
print("Colonnes détectées dans le fichier :", df.columns.tolist()[:15])

# Liste de features adaptées à l'ESP32 avec les noms exacts du CIC-IoT2023
# Note : 'Magnitude' est souvent écrit 'Magnitue' (faute d'orthographe dans le dataset original)
# ou simplement 'magnitude'. Nous allons utiliser une approche flexible.

possible_features = [
    'flow_duration', 'Header_Length', 'Protocol Type',
    'Duration', 'Rate', 'Srate', 'Drate', 'Magnitude', 'Radius', 'Weight'
]

# On ne garde que les colonnes qui existent réellement dans le fichier chargé
selected_features = [f for f in possible_features if f in df.columns]

if 'Magnitude' not in selected_features:
    # Tentative de récupération si le nom est en minuscule
    if 'magnitude' in df.columns:
        selected_features.append('magnitude')
    elif 'Magnitue' in df.columns: # Cas fréquent de la faute de frappe CIC
        selected_features.append('Magnitue')

print(f"Features sélectionnées pour le modèle : {selected_features}")

# Encodage binaire du label
# Dans certaines versions, la colonne est 'label', dans d'autres 'Label'
label_col = 'label' if 'label' in df.columns else 'Label'
df['label_binary'] = df[label_col].apply(lambda x: 0 if 'Benign' in str(x) else 1)

X = df[selected_features]
y = df['label_binary']

# Suppression des valeurs infinies ou NaN (courant dans les captures réseau)
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# -----------------------------------------------------------------
# 4. SPLIT TEMPOREL
# -----------------------------------------------------------------
print("Étape 3 : Split temporel et Mise à l'échelle...")
split_limit = int(len(df) * 0.8)

X_train_raw = X.iloc[:split_limit]
X_test_raw = X.iloc[split_limit:]
y_train = y.iloc[:split_limit]
y_test = y.iloc[split_limit:]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

Étape 2 : Normalisation des colonnes et sélection des features...
Colonnes détectées dans le fichier : ['flow_duration', 'Header_Length', 'Protocol Type', 'Duration', 'Rate', 'Srate', 'Drate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count']
Features sélectionnées pour le modèle : ['flow_duration', 'Header_Length', 'Protocol Type', 'Duration', 'Rate', 'Srate', 'Drate', 'Radius', 'Weight', 'Magnitue']
Étape 3 : Split temporel et Mise à l'échelle...


In [4]:
# -----------------------------------------------------------------
# 4. SPLIT TEMPOREL (Simulé par l'ordre d'arrivée des paquets)
# -----------------------------------------------------------------
print("Étape 2 : Split temporel (80% Train / 20% Test)...")
split_limit = int(len(df) * 0.8)

X_train_raw = X.iloc[:split_limit]
X_test_raw = X.iloc[split_limit:]
y_train = y.iloc[:split_limit]
y_test = y.iloc[split_limit:]

# Normalisation (Fit uniquement sur le Train pour éviter le Data Leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

Étape 2 : Split temporel (80% Train / 20% Test)...


In [ ]:
# -----------------------------------------------------------------
# 5. PIPELINE McINNES : UMAP & HDBSCAN
# -----------------------------------------------------------------
print("Étape 3 : Application du pipeline McInnes pour le raffinement des clusters...")

# Réduction de dimension pour exposer la topologie des attaques
# Nous utilisons 3 composantes pour capturer plus de variance
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=3, random_state=42)
X_train_umap = reducer.fit_transform(X_train)

# Clustering HDBSCAN pour identifier les densités de trafic anormal
clusterer = hdbscan.HDBSCAN(min_cluster_size=50, prediction_data=True)
train_clusters = clusterer.fit_predict(X_train_umap)

# Le Random Forest agit ici comme l'Oracle pour valider la structure apprise
oracle_rf = RandomForestClassifier(n_estimators=50, max_depth=10)
oracle_rf.fit(X_train, y_train)


Étape 3 : Application du pipeline McInnes pour le raffinement des clusters...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
# -----------------------------------------------------------------
# 6. ENTRAÎNEMENT DU MODÈLE TINYML (DISTILLATION)
# -----------------------------------------------------------------
print("Étape 4 : Entraînement du réseau de neurones compact...")

# Structure optimisée pour ESP32 (Memory Footprint réduit)
model = models.Sequential([
    layers.Input(shape=(len(selected_features),)),
    layers.Dense(16, activation='relu'),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
# -----------------------------------------------------------------
# 7. EXPORT TFLITE AVEC QUANTISATION
# -----------------------------------------------------------------
print("Étape 5 : Conversion en TFLite pour microcontrôleur...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT] # Quantisation dynamique
tflite_model = converter.convert()

with open("model_ddos_agri.tflite", "wb") as f:
    f.write(tflite_model)

In [ ]:
# -----------------------------------------------------------------
# 8. ÉVALUATION ET GÉNÉRATION DU RAPPORT FINAL
# -----------------------------------------------------------------
print("\n" + "="*40)
print("RAPPORT FINAL DE PERFORMANCE")
print("="*40)

y_pred = (model.predict(X_test) > 0.5).astype(int)
print(classification_report(y_test, y_pred))

model_size = len(tflite_model) / 1024
print(f"Poids du modèle TFLite : {model_size:.2f} KB")
print(f"Compatibilité ESP32 : {'OUI' if model_size < 200 else 'NÉCESSITE OPTIMISATION'}")

In [ ]:
# -----------------------------------------------------------------
# 9. GÉNÉRATION DU HEADER C POUR DÉPLOIEMENT
# -----------------------------------------------------------------
def generate_c_array(tflite_content, name="model_data"):
    c_str = f"const unsigned char {name}[] = {{\n  "
    for i, val in enumerate(tflite_content):
        c_str += f"0x{val:02x}, "
        if (i + 1) % 12 == 0:
            c_str += "\n  "
    c_str += "\n};\n"
    c_str += f"const unsigned int {name}_len = {len(tflite_content)};"
    return c_str

with open("model_data.h", "w") as f:
    f.write(generate_c_array(tflite_model))

print("\nFichier 'model_data.h' généré avec succès pour votre projet ESP32.")